#**Amazon and flipkart's python specific api snippet**

And get your example outputs for both of them from here

Amazon

https://rapidapi.com/letscrape-6bRBa3QguO5/api/real-time-amazon-data/playground/endpoint_369599f7-6147-4cb9-9417-09dcd429936d

Flipkart

https://rapidapi.com/opendatapoint-opendatapoint-default/api/real-time-flipkart-api/playground/apiendpoint_f88f8fe3-d128-4b50-a6ea-d2020838655a



Go to example response and use that for refference on the output



---





---



#Actual code starts here, put all the pips here

In [ ]:
# Code Snippet 1: Upload Firebase Admin Key to Colab Storage

from google.colab import files
import os

# This will prompt you to upload your Firebase Admin SDK JSON key file.
uploaded = files.upload()

# Assuming you upload only one file, get its filename.
original_key_file = list(uploaded.keys())[0]

# Optionally, rename the file to a known filename for consistent access.
new_key_file_name = 'firebase_admin_key.json'
os.rename(original_key_file, new_key_file_name)

print(f"Firebase Admin key file has been uploaded and stored as: {new_key_file_name}")


Saving capstone-p11-12-firebase-adminsdk-hrk0d-7ffc4a98df.json to capstone-p11-12-firebase-adminsdk-hrk0d-7ffc4a98df.json
Firebase Admin key file has been uploaded and stored as: firebase_admin_key.json


In [ ]:
import http.client
import firebase_admin
from firebase_admin import credentials, firestore
import json
import datetime
import time
import re

# Use the stored Firebase Admin key (ensure the file exists)
key_file_name = 'firebase_admin_key.json'

# Initialize Firebase Admin if not already initialized
if not firebase_admin._apps:
    cred = credentials.Certificate(key_file_name)
    firebase_admin.initialize_app(cred)

db = firestore.client()

# API Credentials for RapidAPI endpoints
AMAZON_API_HOST = "real-time-amazon-data.p.rapidapi.com"
FLIPKART_API_HOST = "real-time-flipkart-api.p.rapidapi.com"
RAPIDAPI_KEY = "f4ffa51701mshde278ef05fe4419p176d56jsn3468a43cf3d9"

def get_price_from_api(source, product_id):
    """Fetches the latest price of a product from the appropriate API."""
    try:
        conn = None
        headers = {
            'x-rapidapi-key': RAPIDAPI_KEY,
            'x-rapidapi-host': AMAZON_API_HOST if source == "Amazon" else FLIPKART_API_HOST
        }

        if source == "Amazon":
            conn = http.client.HTTPSConnection(AMAZON_API_HOST)
            endpoint = f"/product-details?asin={product_id}&country=IN"
        elif source == "Flipkart":
            conn = http.client.HTTPSConnection(FLIPKART_API_HOST)
            endpoint = f"/product-details?pid={product_id}&pincode=110011"
        else:
            print(f"Unsupported source: {source}")
            return None

        conn.request("GET", endpoint, headers=headers)
        res = conn.getresponse()
        data = json.loads(res.read().decode("utf-8"))

        if source == "Amazon":
            raw_price = data.get("data", {}).get("product_price", "Unavailable")
        else:
            raw_price = data.get("price", "Unavailable")

        if isinstance(raw_price, str) and raw_price != "Unavailable":
            numeric_part = re.sub(r"[^\d.]", "", raw_price)
            try:
                price_value = float(numeric_part)
                price = f"₹{price_value:,.2f}"
            except ValueError:
                price = "Unavailable"
        elif isinstance(raw_price, (int, float)):
            price = f"₹{float(raw_price):,.2f}"
        else:
            price = "Unavailable"

        return price
    except Exception as e:
        print(f"Error fetching price for {product_id} from {source}: {e}")
        return None

def update_price_history():
    """Updates price history using date as document ID in priceEntries subcollection."""
    users_ref = db.collection('users')
    price_history_ref = db.collection('priceHistory')
    processed_products = set()

    for user_doc in users_ref.stream():
        tracked_products_ref = user_doc.reference.collection('trackedProducts')

        for product_doc in tracked_products_ref.stream():
            product_data = product_doc.to_dict()
            product_id = product_data.get('asinOrPid')
            source = product_data.get('source')

            if not product_id or not source:
                continue

            if product_id in processed_products:
                continue
            processed_products.add(product_id)

            latest_price = get_price_from_api(source, product_id)
            if not latest_price:
                continue

            # Create/update product document
            product_doc_ref = price_history_ref.document(product_id)
            if not product_doc_ref.get().exists:
                product_doc_ref.set({
                    'source': source,
                    'product_id': product_id,
                    'name': product_data.get('productName', ''),
                    'url': product_data.get('productUrl', '')
                })

            # Create price entry with date as document ID
            formatted_date = datetime.datetime.utcnow().strftime("%d-%m-%Y")  # Use %Y for four-digit year
            price_entry = {
                'price': latest_price,
                'source': source,
                'user_id': user_doc.id
            }

            # Use date string as document ID
            price_entry_ref = product_doc_ref.collection('priceEntries').document(formatted_date)
            if not price_entry_ref.get().exists:
                price_entry_ref.set(price_entry)
                print(f"Logged {formatted_date}: {latest_price} ({source})")
            else:
                print(f"Entry for {formatted_date} already exists, skipping")

# Run the price tracking update periodically
if __name__ == "__main__":
        update_price_history()
        print("Waiting for next update...")
#        time.sleep(86400)  # Check every 24 hours

Logged 21-03-2025: ₹45,490.00 (Amazon)
Logged 21-03-2025: ₹63,990.00 (Amazon)
Logged 21-03-2025: ₹18,824.00 (Amazon)
Logged 21-03-2025: ₹112,900.00 (Amazon)
Logged 21-03-2025: ₹116,999.00 (Amazon)
Logged 21-03-2025: ₹97,990.00 (Flipkart)
Logged 21-03-2025: ₹76,990.00 (Flipkart)
Logged 21-03-2025: ₹64,999.00 (Flipkart)
Logged 21-03-2025: ₹78,999.00 (Flipkart)
Logged 21-03-2025: ₹40,999.00 (Flipkart)
Waiting for next update...


In [ ]:
import firebase_admin
from firebase_admin import credentials, firestore
import pandas as pd
import re
import datetime

# ETS Libraries
from statsmodels.tsa.exponential_smoothing.ets import ETSModel

from statsmodels.tsa.stattools import adfuller

# Initialize Firebase Admin
key_file_name = 'firebase_admin_key.json'
if not firebase_admin._apps:
    cred = credentials.Certificate(key_file_name)
    firebase_admin.initialize_app(cred)

db = firestore.client()

def is_stationary(series):
    """Check if a time series is stationary using the ADF test."""
    result = adfuller(series)
    return result[1] <= 0.05

def fetch_price_history(product_id):
    """Fetches and processes historical price data from Firestore."""
    price_history_ref = db.collection('priceHistory').document(product_id).collection('priceEntries')
    docs = price_history_ref.stream()

    data = []
    for doc in docs:
        doc_data = doc.to_dict()
        date_str = doc.id

        # Convert price string to float
        raw_price = doc_data.get("price", "Unavailable")
        if raw_price != "Unavailable":
            try:
                price_value = float(re.sub(r"[^\d.]", "", raw_price))
            except ValueError:
                continue

            # Parse date
            try:
                date_obj = datetime.datetime.strptime(date_str, "%d-%m-%Y")
                data.append({"date": date_obj, "price": price_value})
            except ValueError:
                continue

    if not data:
        print(f"No valid data for {product_id}")
        return None, False

    # Create and clean DataFrame
    df = pd.DataFrame(data)
    df.sort_values("date", inplace=True)
    df.set_index("date", inplace=True)

    # Resample to daily frequency and forward fill
    df = df.resample('D').first().ffill()

    stationary = is_stationary(df['price'])
    return df, stationary

# ------------------------ ETS Model ------------------------
def predict_next_30_days_ets(df):
    """Generates ETS forecast for the next 30 days."""
    if df is None or df.shape[0] < 30:
        print(f"Insufficient data for ETS: {df.shape[0] if df else 0} points")
        return None

    try:
        model = ETSModel(df["price"], error='add', trend='add', seasonal='add', seasonal_periods=30)
        model_fit = model.fit()
        forecast = model_fit.forecast(steps=30)
        return forecast
    except Exception as e:
        print(f"ETS modeling failed: {str(e)}")
        return None

# ------------------------ Storage and Execution ------------------------
def store_predicted_prices(product_id, forecast):
    """Stores predictions in Firestore with proper formatting."""
    if forecast is None:
        return

    predictions = []
    for date_index, value in forecast.items():
        predictions.append({
            "date": pd.to_datetime(date_index).strftime("%d-%m-%Y"),
            "price": f"₹{value:,.2f}"
        })

    db.collection('priceHistory').document(product_id).set(
        {"predictedPrice": predictions},
        merge=True
    )

def delete_old_predictions(product_id):
    """Clears previous predictions."""
    db.collection('priceHistory').document(product_id).update({
        "predictedPrice": firestore.DELETE_FIELD
    })

def run_predictions(): # Removed model_type parameter
    """Main execution flow for price predictions using ETS.""" # Updated docstring
    products_ref = db.collection('priceHistory').stream()

    for product in products_ref:
        product_id = product.id
        print(f"Processing {product_id} with ETS...") # Updated print statement

        delete_old_predictions(product_id)
        df, is_stationary = fetch_price_history(product_id)
        if df is None:
            continue

        forecast = predict_next_30_days_ets(df) # Directly call ETS function
        if forecast is not None:
            store_predicted_prices(product_id, forecast)

if __name__ == "__main__":
    run_predictions() # Removed model_to_run selection
    print(f"Prediction cycle completed using ETS") # Updated print statement

Processing B0CX5FRD9H with ETS...
Processing B0D5DFR78J with ETS...
Processing B0DDTXNGYN with ETS...
Processing B0DGJC8DG8 with ETS...
Processing B0DSBTKP5Q with ETS...
Processing COMGYP5GGSNGYSZP with ETS...
Processing COMHYWSWHTGX8BWG with ETS...
Processing MOBGTAGPTB3VS24W with ETS...
Processing MOBH4DQFHZ7NFCME with ETS...
Processing TVSH3YBHMBFYQXRV with ETS...
Prediction cycle completed using ETS


In [ ]:
import firebase_admin
from firebase_admin import credentials, firestore
import pandas as pd
import re
import datetime
from statsmodels.tsa.exponential_smoothing.ets import ETSModel

# Firebase Initialization
key_file_name = 'firebase_admin_key.json'
if not firebase_admin._apps:
    cred = credentials.Certificate(key_file_name)
    firebase_admin.initialize_app(cred)
db = firestore.client()

def fetch_price_history(product_id):
    """Fetches price history from Firestore."""
    docs = db.collection('priceHistory').document(product_id).collection('priceEntries').stream()
    data = []
    for doc in docs:
        doc_data = doc.to_dict()
        raw_price = doc_data.get("price", "Unavailable")
        if raw_price != "Unavailable":
            try:
                price_value = float(re.sub(r"[^\d.]", "", raw_price))
                date_obj = datetime.datetime.strptime(doc.id, "%d-%m-%Y")
                data.append({"date": date_obj, "price": price_value})
            except (ValueError, TypeError): # Catching both Value and Type errors in one go
                continue
    if not data:
        print(f"No valid data for {product_id}")
        return None
    df = pd.DataFrame(data).sort_values("date").set_index("date").resample('D').first().ffill()
    return df

def predict_price_ets(df, product_id):
    """Predicts next 30 days price using ETS."""
    if df is None or len(df) < 30:
        print(f"Data too short for ETS {product_id}")
        return None
    try:
        model = ETSModel(df['price'], error='add', trend='add', seasonal='add', seasonal_periods=7).fit()
        forecast = model.forecast(steps=30)
        return pd.Series(forecast, index=pd.date_range(df.index[-1] + datetime.timedelta(days=1), periods=30, freq='D'))
    except Exception as e:
        print(f"ETS fail {product_id}: {e}")
        return None

def generate_buy_advice(historical_df, forecast_prices):
    """Generates buy advice tailored for e-commerce advisory with refined advice."""
    if historical_df is None or forecast_prices is None:
        return "No Advice", None, None, None
    last_price = historical_df['price'].iloc[-1]
    predicted_price_next_day = forecast_prices.iloc[0]
    avg_month_price = forecast_prices.mean()
    day_increase_perc = (predicted_price_next_day - last_price) / last_price * 100
    month_increase_perc = (avg_month_price - last_price) / last_price * 100

    if month_increase_perc > 5 and day_increase_perc > 2:
        advice = "Absolutely Buy: Strong price increase expected soon."
    elif month_increase_perc < -3 or (month_increase_perc < 1 and day_increase_perc < -1):
        advice = "Wait if Possible: Price likely to decrease or remain low."
    elif month_increase_perc > 1 and day_increase_perc >= 0:
        advice = "Maybe Buy: Price is trending up, consider purchasing."
    else:
        advice = "Not Now: Monitor price trends before deciding."

    return advice, last_price, predicted_price_next_day, avg_month_price

def store_advice_firestore(product_id, advice, forecast=None, last_price=None, next_day_price=None, avg_month_price=None):
    """Stores buy advice and prices in Firestore."""
    data = {"buyAdvice": advice}
    if forecast is not None:
        data["predictedPrice"] = [{"date": date.strftime("%d-%m-%Y"), "price": f"₹{price:,.2f}"} for date, price in forecast.items()]
    if last_price: data["lastHistoricalPrice"] = f"₹{last_price:,.2f}"
    if next_day_price: data["predictedPriceNextDay"] = f"₹{next_day_price:,.2f}"
    if avg_month_price: data["avgPredictedPriceNextMonth"] = f"₹{avg_month_price:,.2f}"
    db.collection('priceHistory').document(product_id).set(data, merge=True)

def run_buy_advice_cycle_ets():
    """Main function to run buy advice cycle using ETS."""
    products_ref = db.collection('priceHistory').stream()
    for product_doc in products_ref:
        product_id = product_doc.id
        print(f"ETS Advice for {product_id}...")
        historical_df = fetch_price_history(product_id)
        if historical_df is None: continue
        forecast = predict_price_ets(historical_df, product_id)
        advice, last_price, next_day_price, avg_month_price = generate_buy_advice(historical_df, forecast)
        print(f"Product: {product_id}, Last Price: ₹{last_price:.2f}, Next Day: ₹{next_day_price:.2f}, Avg Month: ₹{avg_month_price:.2f}, Advice: {advice}")
        store_advice_firestore(product_id, advice, forecast, last_price, next_day_price, avg_month_price)

if __name__ == "__main__":
    run_buy_advice_cycle_ets()
    print("ETS Buy advice cycle completed.")

ETS Advice for B0CX5FRD9H...
Product: B0CX5FRD9H, Last Price: ₹45490.00, Next Day: ₹45655.37, Avg Month: ₹44973.67, Advice: Not Now: Monitor price trends before deciding.
ETS Advice for B0D5DFR78J...
Product: B0D5DFR78J, Last Price: ₹63990.00, Next Day: ₹64284.00, Avg Month: ₹64049.60, Advice: Not Now: Monitor price trends before deciding.
ETS Advice for B0DDTXNGYN...
Product: B0DDTXNGYN, Last Price: ₹18824.00, Next Day: ₹18424.91, Avg Month: ₹18706.11, Advice: Wait if Possible: Price likely to decrease or remain low.
ETS Advice for B0DGJC8DG8...
Product: B0DGJC8DG8, Last Price: ₹112900.00, Next Day: ₹112642.07, Avg Month: ₹112777.12, Advice: Not Now: Monitor price trends before deciding.
ETS Advice for B0DSBTKP5Q...
Product: B0DSBTKP5Q, Last Price: ₹116999.00, Next Day: ₹117694.67, Avg Month: ₹118004.42, Advice: Not Now: Monitor price trends before deciding.
ETS Advice for COMGYP5GGSNGYSZP...
Product: COMGYP5GGSNGYSZP, Last Price: ₹97990.00, Next Day: ₹98129.43, Avg Month: ₹97855.70, 

In [ ]:
import firebase_admin
from firebase_admin import credentials, firestore, messaging
import time
import re  # Import the regular expression library

# Firebase Initialization (ensure you have firebase_admin_key.json)
key_file_name = 'firebase_admin_key.json'
if not firebase_admin._apps:
	cred = credentials.Certificate(key_file_name)
	firebase_admin.initialize_app(cred)

db = firestore.client()

def get_price_drop_products():
	"""Retrieves products, updates price from priceHistory, and checks for price drops."""
	try:
		price_drop_products = [] # List to store products with price drops
		users_ref = db.collection('users')

		for user_doc in users_ref.stream():
			tracked_products_ref = user_doc.reference.collection('trackedProducts')
			for product_doc in tracked_products_ref.stream():
				product_data = product_doc.to_dict()
				source = product_data.get('source', 'Unknown')
				set_price_str = product_data.get('setTrackingPrice', '0') # Get set price as string
				product_name = product_data.get('productName', 'Product Name Not Found') # Get productName
				product_id_for_history = product_doc.id # Assuming product_doc.id can be used for priceHistory (e.g., 'B0DSBTKP5Q')

				# --- Fetch and Update Price from priceHistory ---
				price_history_ref = db.collection('priceHistory').document(product_id_for_history) # Use product ID to construct path
				price_history_doc = price_history_ref.get()

				if price_history_doc.exists:
					price_history_data = price_history_doc.to_dict()
					historical_price_str = price_history_data.get('lastHistoricalPrice', None) # Get lastHistoricalPrice (corrected field name)

					if historical_price_str is not None:
						# --- Price Cleaning ---
						cleaned_price_str = re.sub(r'[₹,]', '', historical_price_str).strip() # Remove ₹ and commas
						# --- End Price Cleaning ---

						try:
							historical_price = float(cleaned_price_str)
							# Update the 'price' in trackedProducts with lastHistoricalPrice
							product_doc.reference.update({'price': str(historical_price)}) # Store as string in Firestore if it was originally a string

							product_data['price'] = str(historical_price) # Update local product_data for comparison
							product_price_str = str(historical_price) # Update for price drop check and display

							print(f"  Updated price for product {product_doc.id} from priceHistory: {historical_price}")

						except ValueError:
							print(f"  Error: Could not convert lastHistoricalPrice '{cleaned_price_str}' (after cleaning '{historical_price_str}') to float for product {product_doc.id}")
							continue # Skip this product if price conversion fails for historical price
					else:
						print(f"  Warning: lastHistoricalPrice not found in priceHistory for product {product_doc.id}")
						product_price_str = product_data.get('price', '0.0') # Fallback to existing price if history price missing
				else:
					print(f"  Warning: priceHistory document not found for product {product_doc.id}")
					product_price_str = product_data.get('price', '0.0') # Fallback to existing price if history document missing
				# --- End of Fetch and Update Price from priceHistory ---


				try:
					price = float(product_price_str) # Use price from trackedProducts (potentially updated from priceHistory)
					set_price = float(set_price_str)
				except ValueError:
					print(f"  Error: Could not convert price or set price to float for product {product_doc.id}")
					continue # Skip this product if price conversion fails

				formatted_price = f"₹{price:,.0f}"
				formatted_set_price = f"₹{set_price:,.0f}"

				# Check if current price in trackedProducts is lower than set price
				if price < set_price:
					print(f"    Price Drop Alert! Source: {source}, Product Name: {product_name}, Current Price: {formatted_price}, Set Price: {formatted_set_price}") # Added Product Name to print
					price_drop_products.append({
						'user_id': user_doc.id,  # Add user ID to product data
						'source': source,
						'product_name': product_name, # Add product name to product data
						'current_price': formatted_price,  # Use formatted price from trackedProducts (updated)
						'set_price': formatted_set_price
					})
				else:
					print(f"    Source: {source}, Product Name: {product_name}, Price: {formatted_price}, Set Price: {formatted_set_price} - No Drop") # Added Product Name to print

		return price_drop_products # Return list of products with price drops
	except Exception as e:
		print(f"Error getting tracked products: {e}")
		return []  # Return empty list in case of error


def send_price_drop_notifications(price_drop_products):
	"""Sends notifications for products that have dropped below their set tracking price."""
	try:
		if not price_drop_products:
			print("No price drop alerts to send.")
			return

		users_ref = db.collection('users') # Re-fetch users collection (if needed)

		for product in price_drop_products:
			user_id = product['user_id']
			user_doc = users_ref.document(user_id).get() # Get specific user doc
			if not user_doc.exists:
				print(f"Warning: User document {user_id} not found, skipping notification.")
				continue

			user_data = user_doc.to_dict()
			tokens = user_data.get('fcmTokens', [])

			if not tokens:
				print(f"No FCM tokens for user: {user_id}")  # Debug: No tokens
				continue

			# Build the price drop message and notification
			notification_title = "Price Drop Alert!"
			message_body = f"Product: {product['product_name']} is now at {product['current_price']}, below your set price of {product['set_price']}!"

			# Send to each token as DATA MESSAGE instead of NOTIFICATION MESSAGE
			for token in tokens:
				data_message = { # Changed to data message format
					"title": notification_title,
					"body": message_body.strip(),
					"product_name": product['product_name'],
					"current_price": product['current_price'],
					"set_price": product['set_price']
				}

				message = messaging.Message(
					data=data_message, # Sending data message instead of notification
					token=token
				)
				try:
					response = messaging.send(message)
					print(f"Sent price drop notification as data message to {token}: {response}")  # Debug: Sent message
				except Exception as e:
					print(f"Failed to send price drop notification as data message to {token}: {e}") # Debug: Send failure

	except Exception as e:
		print(f"Error sending price drop notifications: {e}")  # Debug: General error


# Main loop for checking and sending notifications
if __name__ == "__main__":
	price_drop_products = get_price_drop_products()  # Get price drop products (now updates price too)
	if price_drop_products:
		send_price_drop_notifications(price_drop_products) # Send notifications for price drops
	else:
		print("No products with price drops found.")

	print("Price check and notifications completed. Exiting.")

  Updated price for product B0CX5FRD9H from priceHistory: 45490.0
    Price Drop Alert! Source: Amazon, Product Name: Samsung 138 cm (55 inches) D Series Brighter Crystal 4K Vivid Pro Ultra HD Smart LED TV UA55DUE77AKLXL (Black), Current Price: ₹45,490, Set Price: ₹400,000
  Updated price for product B0D5DFR78J from priceHistory: 63990.0
    Source: Amazon, Product Name: ASUS TUF Gaming A15, AMD Ryzen 7 7435Hs, 15.6-Inch, FHD 144Hz, Gaming Laptop (16GB RAM/512GB SSD/NVIDIA Geforce RTX 3050/Windows 11/48WHR/Graphite Black/2.3 Kg), FA506NCR-HN054W, Price: ₹63,990, Set Price: ₹60,000 - No Drop
  Updated price for product B0DDTXNGYN from priceHistory: 18824.0
    Source: Amazon, Product Name: realme NARZO 70 Turbo 5G (Turbo Purple,8GB RAM,128GB Storage) | Segment's Fastest Dimensity 7300 Energy 5G Chipset | Motorsports Inspired Design, Price: ₹18,824, Set Price: ₹7,000 - No Drop
  Updated price for product B0DGJC8DG8 from priceHistory: 112900.0
    Price Drop Alert! Source: Amazon, Product